## Split Original Real-world Data into Train and Test Sets

In [1]:
import pandas as pd
import numpy as np
import gzip
from pathlib import Path
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print("🔄 Source-Specific Balanced Dataset Split")
print("=" * 70)

# =============================================================================
# CONFIGURATION
# =============================================================================

INPUT_FILE      = "../raw/five_email_phishing.csv.gz"
OUTPUT_DIR      = "../raw/"

TRAIN_SIZE      = 1000  # per source
TEST_SIZE       = 1000  # per source
RANDOM_STATE    = 42

DROP_MISSING    = True
MISSING_THRESHOLD = 0.0

# =============================================================================
# 1. LOAD & CLEAN DATA
# =============================================================================

def load_data(path):
    with gzip.open(path, 'rt', encoding='utf-8') as f:
        return pd.read_csv(f)

print("📂 Loading data...")
df = load_data(INPUT_FILE)
print(f"Original data shape: {df.shape}")

# Apply your cleaning logic here (assuming df_clean is the result)
# For now, using df as df_clean - replace with your actual cleaning code
df_clean = df.copy()

# =============================================================================
# 2. SOURCE ANALYSIS
# =============================================================================

print("\n🔍 Analyzing source distribution:")
source_counts = df_clean['source'].value_counts()
print(source_counts.to_string())

print(f"\n📊 Label distribution per source:")
for source in source_counts.index:
    source_data = df_clean[df_clean['source'] == source]
    label_dist = source_data['label'].value_counts()
    benign_count = label_dist.get(0, 0)
    malicious_count = label_dist.get(1, 0)
    total = len(source_data)
    
    print(f"\n  {source}:")
    print(f"    Total: {total}")
    print(f"    Benign (0): {benign_count}")
    print(f"    Malicious (1): {malicious_count}")
    
    # Check if we have enough samples for balanced split
    min_class_size = min(benign_count, malicious_count)
    samples_per_class = (TRAIN_SIZE + TEST_SIZE) // 2
    
    if min_class_size >= samples_per_class:
        print(f"    ✅ Sufficient data (need {samples_per_class} per class, have {min_class_size})")
    else:
        print(f"    ⚠️  Insufficient data (need {samples_per_class} per class, have {min_class_size})")

# =============================================================================
# 3. SOURCE-SPECIFIC BALANCED SPLIT FUNCTION
# =============================================================================

def create_balanced_split(df, source_name, train_size, test_size, label_col='label', random_state=42):
    """
    Create balanced train/test split for a specific source.
    """
    # Filter data for this source
    source_data = df[df['source'] == source_name].copy()
    
    # Get samples for each class
    benign_samples = source_data[source_data[label_col] == 0]
    malicious_samples = source_data[source_data[label_col] == 1]
    
    # Calculate samples needed per class
    train_per_class = train_size // 2
    test_per_class = test_size // 2
    total_per_class = train_per_class + test_per_class
    
    # Check if we have enough samples
    if len(benign_samples) < total_per_class or len(malicious_samples) < total_per_class:
        available_benign = len(benign_samples)
        available_malicious = len(malicious_samples)
        print(f"    ❌ Skipping {source_name}: insufficient samples")
        print(f"       Need {total_per_class} per class, have {available_benign} benign, {available_malicious} malicious")
        return None, None
    
    # Sample equal amounts from each class
    benign_sample = benign_samples.sample(n=total_per_class, random_state=random_state)
    malicious_sample = malicious_samples.sample(n=total_per_class, random_state=random_state)
    
    # Split into train/test for each class
    benign_train = benign_sample.iloc[:train_per_class]
    benign_test = benign_sample.iloc[train_per_class:train_per_class + test_per_class]
    
    malicious_train = malicious_sample.iloc[:train_per_class]
    malicious_test = malicious_sample.iloc[train_per_class:train_per_class + test_per_class]
    
    # Combine and shuffle
    train_df = pd.concat([benign_train, malicious_train]).sample(frac=1, random_state=random_state).reset_index(drop=True)
    test_df = pd.concat([benign_test, malicious_test]).sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    return train_df, test_df

# =============================================================================
# 4. PROCESS EACH SOURCE
# =============================================================================

def save_gzip(df, path):
    """Save DataFrame as gzipped CSV"""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(path, 'wt', encoding='utf-8') as f:
        df.to_csv(f, index=False)
    size_mb = Path(path).stat().st_size / 1024**2
    return size_mb

print(f"\n🚀 Processing each source (targeting {TRAIN_SIZE} train + {TEST_SIZE} test per source):")
print("=" * 70)

successful_sources = []
failed_sources = []

for source in source_counts.index:
    print(f"\n📁 Processing source: {source}")
    
    # Create balanced split
    train_df, test_df = create_balanced_split(
        df_clean, source, TRAIN_SIZE, TEST_SIZE, random_state=RANDOM_STATE
    )
    
    if train_df is not None and test_df is not None:
        # Clean source name for filename (replace problematic characters)
        clean_source = source.replace('/', '_').replace('\\', '_').replace(' ', '_')
        
        # Define output paths
        train_path = f"{OUTPUT_DIR}email_phishing_{clean_source}_train.csv.gz"
        test_path = f"{OUTPUT_DIR}email_phishing_{clean_source}_test.csv.gz"
        
        # Save files
        train_size_mb = save_gzip(train_df, train_path)
        test_size_mb = save_gzip(test_df, test_path)
        
        # Verify label distribution
        train_label_dist = train_df['label'].value_counts()
        test_label_dist = test_df['label'].value_counts()
        
        print(f"    ✅ Successfully created splits:")
        print(f"       Train: {len(train_df)} samples ({train_size_mb:.2f} MB)")
        print(f"              Benign: {train_label_dist.get(0, 0)}, Malicious: {train_label_dist.get(1, 0)}")
        print(f"       Test:  {len(test_df)} samples ({test_size_mb:.2f} MB)")
        print(f"              Benign: {test_label_dist.get(0, 0)}, Malicious: {test_label_dist.get(1, 0)}")
        print(f"       Files: {train_path}")
        print(f"              {test_path}")
        
        successful_sources.append(source)
    else:
        failed_sources.append(source)

# =============================================================================
# 5. SUMMARY
# =============================================================================

print("\n" + "=" * 70)
print("🎉 PROCESSING COMPLETE")
print("=" * 70)

print(f"\n✅ Successfully processed {len(successful_sources)} sources:")
for source in successful_sources:
    clean_source = source.replace('/', '_').replace('\\', '_').replace(' ', '_')
    print(f"   - {source} → email_phishing_{clean_source}_{{train|test}}.csv.gz")

if failed_sources:
    print(f"\n❌ Failed to process {len(failed_sources)} sources (insufficient data):")
    for source in failed_sources:
        print(f"   - {source}")

print(f"\n📁 All files saved to: {OUTPUT_DIR}")
print(f"📊 Per source: {TRAIN_SIZE} train + {TEST_SIZE} test samples (balanced)")

🔄 Source-Specific Balanced Dataset Split
📂 Loading data...
Original data shape: (131346, 4)

🔍 Analyzing source distribution:
source
TREC-07     53757
CEAS-08     39154
Enron       29767
Assassin     5809
Ling         2859

📊 Label distribution per source:

  TREC-07:
    Total: 53757
    Benign (0): 24358
    Malicious (1): 29399
    ✅ Sufficient data (need 1000 per class, have 24358)

  CEAS-08:
    Total: 39154
    Benign (0): 17312
    Malicious (1): 21842
    ✅ Sufficient data (need 1000 per class, have 17312)

  Enron:
    Total: 29767
    Benign (0): 15791
    Malicious (1): 13976
    ✅ Sufficient data (need 1000 per class, have 13976)

  Assassin:
    Total: 5809
    Benign (0): 4091
    Malicious (1): 1718
    ✅ Sufficient data (need 1000 per class, have 1718)

  Ling:
    Total: 2859
    Benign (0): 2401
    Malicious (1): 458
    ⚠️  Insufficient data (need 1000 per class, have 458)

🚀 Processing each source (targeting 1000 train + 1000 test per source):

📁 Processing source